In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/__results__.html
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/__notebook__.ipynb
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/__output__.json
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/custom.css
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D2.csv
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/class_balance_verification.csv
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D3.csv
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D12.csv
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D1.csv
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D23.csv
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D123.csv
/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/

In [2]:
!pip install -q peft timm huggingface_hub
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
import torch, timm, gc
from huggingface_hub import login, list_repo_files, hf_hub_download
from kaggle_secrets import UserSecretsClient
login(UserSecretsClient().get_secret("HF_TOKEN"))

REPO = "YukunZhou/RETFound_mae_natureCFP"
files = list_repo_files(REPO)
wfile = [f for f in files if f.endswith((".pth", ".bin", ".safetensors"))][0]
path = hf_hub_download(REPO, wfile)

try:    ck = torch.load(path, map_location="cpu")
except Exception: ck = torch.load(path, map_location="cpu", weights_only=False)
sd = ck.get("model", ck) if isinstance(ck, dict) else ck

sd = {k: v for k, v in sd.items() if not k.startswith(("decoder", "mask_token", "head"))}
sd["fc_norm.weight"], sd["fc_norm.bias"] = sd.pop("norm.weight"), sd.pop("norm.bias")

m = timm.create_model("vit_large_patch16_224", pretrained=False, num_classes=0,
                      global_pool="avg", img_size=224)
print(m.load_state_dict(sd, strict=False))
print("params (M):", sum(p.numel() for p in m.parameters())/1e6)
del ck, sd; gc.collect()

RETFound_mae_natureCFP.pth: reconstructing file:   0%|          |  0.00B / 3.95GB            

RETFound_mae_natureCFP.pth: downloading bytes:           |  0.00B            

<All keys matched successfully>
params (M): 303.301632


105

In [4]:
import os, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import f1_score
import peft.tuners.lora.torchao as _pt
_pt.is_torchao_available = lambda: False
from peft import LoraConfig, get_peft_model

T_START = time.time()
def status(msg):
    print(f"[STATUS {(time.time()-T_START)/60:5.1f} min] {msg}", flush=True)

MODE = "xattn_query"        # "vit" | "concat" | "xattn" | "xattn_query"
CNN_NAME = "resnet50"
CSV = "/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D123.csv"
LABELS = ["Normal","DR","Glaucoma","AMD","Myopia"]
IMG, VIT_IMG, BS, EPOCHS, PATIENCE = 320, 224, 16, 30, 7
NUM_WORKERS = 2             # set to 0 if the run stalls
SEED = 42
dev = "cuda"; TAG = f"hybrid_{MODE}"
torch.manual_seed(SEED); np.random.seed(SEED)
status(f"STARTING | mode={MODE} | cnn={CNN_NAME} | epochs={EPOCHS} | gpu={torch.cuda.get_device_name(0)}")

# ---------------- data ----------------
status("Step 1/5: reading CSV and checking that all image files exist")
df = pd.read_csv(CSV)
missing = [p for p in df.image_path if not os.path.exists(p)]
assert not missing, f"{len(missing)} image files not found, e.g. {missing[:3]}"
tr, va, te = [df[df.split==s].reset_index(drop=True) for s in ["train","val","test"]]
status(f"Data OK | train {len(tr)} | val {len(va)} | test {len(te)}")

mean, std = (0.485,0.456,0.406), (0.229,0.224,0.225)
tf_tr = T.Compose([T.Resize((IMG,IMG)), T.RandomHorizontalFlip(), T.RandomRotation(15),
                   T.ColorJitter(0.2,0.2), T.ToTensor(), T.Normalize(mean,std)])
tf_ev = T.Compose([T.Resize((IMG,IMG)), T.ToTensor(), T.Normalize(mean,std)])

class DS(Dataset):
    def __init__(s, d, tf): s.d, s.tf = d, tf
    def __len__(s): return len(s.d)
    def __getitem__(s, i):
        r = s.d.iloc[i]
        x = s.tf(Image.open(r.image_path).convert("RGB"))
        return x, torch.tensor(r[LABELS].values.astype("float32"))
mk = lambda d, tf, sh: DataLoader(DS(d, tf), BS, shuffle=sh, num_workers=NUM_WORKERS, pin_memory=True)
dl_tr, dl_va, dl_te = mk(tr,tf_tr,True), mk(va,tf_ev,False), mk(te,tf_ev,False)

# ---------------- model ----------------
status("Step 2/5: building model (RETFound + LoRA, ResNet50, fusion layers)")
vit = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["qkv"]))
vit.print_trainable_parameters()
cnn = timm.create_model(CNN_NAME, pretrained=True, num_classes=0, global_pool="")

def ffn(D): return nn.Sequential(nn.Linear(D,4*D), nn.GELU(), nn.Dropout(0.1), nn.Linear(4*D,D))

class Hybrid(nn.Module):
    def __init__(s, vit, cnn, mode, D=512, ncls=5, n_tok=100):
        super().__init__()
        s.vit, s.cnn, s.mode = vit, cnn, mode
        dv, dc = 1024, cnn.num_features
        if mode == "vit":
            s.head = nn.Sequential(nn.LayerNorm(dv), nn.Dropout(0.3), nn.Linear(dv, ncls))
        elif mode == "concat":
            s.head = nn.Sequential(nn.LayerNorm(dv+dc), nn.Dropout(0.3), nn.Linear(dv+dc, ncls))
        else:
            s.pv, s.pc = nn.Linear(dv, D), nn.Linear(dc, D)
            s.pos_c = nn.Parameter(torch.randn(1, n_tok, D) * .02)
            s.ln_q, s.ln_kv, s.ln_f = nn.LayerNorm(D), nn.LayerNorm(D), nn.LayerNorm(D)
            s.xattn = nn.MultiheadAttention(D, 8, dropout=0.1, batch_first=True)
            s.ffn = ffn(D)
            if mode == "xattn":
                s.head = nn.Sequential(nn.LayerNorm(D), nn.Dropout(0.3), nn.Linear(D, ncls))
            else:
                s.q = nn.Parameter(torch.randn(ncls, D) * .02)
                s.ln_t, s.ln_d, s.ln_d2 = nn.LayerNorm(D), nn.LayerNorm(D), nn.LayerNorm(D)
                s.dec = nn.MultiheadAttention(D, 8, dropout=0.1, batch_first=True)
                s.ffn_d = ffn(D)
                s.cls_w = nn.Parameter(torch.randn(ncls, D) * .02)
                s.cls_b = nn.Parameter(torch.zeros(ncls))
    def forward(s, x):
        xv = F.interpolate(x, size=VIT_IMG, mode="bilinear", align_corners=False)
        tv = s.vit.base_model.model.forward_features(xv)          # (B,197,1024)
        if s.mode == "vit":
            return s.head(tv[:,1:].mean(1))
        fm = s.cnn.forward_features(x)                             # (B,C,10,10)
        if s.mode == "concat":
            return s.head(torch.cat([tv[:,1:].mean(1), fm.mean((2,3))], 1))
        L = s.pc(fm.flatten(2).transpose(1,2))
        assert L.shape[1] == s.pos_c.shape[1], f"CNN tokens {L.shape[1]} != 100 (use IMG=320)"
        L = s.ln_kv(L + s.pos_c)
        Tk = s.pv(tv)
        a, _ = s.xattn(s.ln_q(Tk), L, L)
        Tk = Tk + a
        Tk = Tk + s.ffn(s.ln_f(Tk))
        if s.mode == "xattn":
            return s.head(Tk.mean(1))
        Q = s.q.unsqueeze(0).expand(x.size(0), -1, -1)
        kv = s.ln_t(Tk)
        d, _ = s.dec(s.ln_d(Q), kv, kv)
        Q = Q + d
        Q = Q + s.ffn_d(s.ln_d2(Q))
        return (Q * s.cls_w).sum(-1) + s.cls_b

model = Hybrid(vit, cnn, MODE).to(dev)
status(f"Model ready | trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f} M")

# ---------------- loss / optimizer ----------------
status("Step 3/5: setting up loss and optimizer")
cnt = tr[LABELS].sum().values
cw = torch.tensor(len(tr)/(5*cnt), dtype=torch.float32, device=dev)
class Focal(nn.Module):
    def __init__(s, a, g=2.0): super().__init__(); s.a, s.g = a, g
    def forward(s, lo, y):
        bce = F.binary_cross_entropy_with_logits(lo, y, reduction="none")
        p = torch.exp(-bce)
        return (s.a * (1-p)**s.g * bce).mean()
crit = Focal(cw)

lora_p = [p for p in model.vit.parameters() if p.requires_grad]
cnn_p = list(model.cnn.parameters())
used = {id(p) for p in lora_p + cnn_p}
new_p = [p for p in model.parameters() if p.requires_grad and id(p) not in used]
opt = torch.optim.Adam([{"params": lora_p, "lr": 3e-4},
                        {"params": cnn_p,  "lr": 1e-4},
                        {"params": new_p,  "lr": 1e-3}])
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
scaler = torch.amp.GradScaler("cuda")

@torch.no_grad()
def probs(dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast("cuda", dtype=torch.float16):
            P.append(torch.sigmoid(model(x.to(dev))).float().cpu().numpy())
        Y.append(y.numpy())
    return np.vstack(P), np.vstack(Y)
macro_f1 = lambda P, Y, th=0.5: f1_score(Y, (P>=th).astype(int), average="macro", zero_division=0)

# ---------------- train ----------------
status(f"Step 4/5: TRAINING STARTED ({EPOCHS} epochs, expect about 3-4 min per epoch)")
best, bad = 0, 0
for ep in range(EPOCHS):
    t_ep = time.time(); model.train(); tot = 0
    for i, (x, y) in enumerate(dl_tr):
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad()
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x)
        loss = crit(out.float(), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tot += loss.item()
        if i % 50 == 0:
            status(f"  epoch {ep+1}/{EPOCHS} | batch {i}/{len(dl_tr)} | loss {loss.item():.4f}")
    status(f"  epoch {ep+1}/{EPOCHS} | training done, validating...")
    P, Y = probs(dl_va); f = macro_f1(P, Y); sched.step(f)
    ep_min = (time.time()-t_ep)/60
    left = ep_min * (EPOCHS-ep-1)
    status(f"EPOCH {ep+1}/{EPOCHS} DONE | loss {tot/len(dl_tr):.4f} | val Macro-F1 {f:.4f} | best {max(best,f):.4f} | {ep_min:.1f} min/epoch | ~{left:.0f} min left (max)")
    if f > best:
        best, bad = f, 0; torch.save(model.state_dict(), f"{TAG}_best.pth")
        status(f"  new best model saved (val Macro-F1 {best:.4f})")
    else:
        bad += 1
        if bad >= PATIENCE:
            status(f"EARLY STOPPING (no improvement for {PATIENCE} epochs)"); break

# ---------------- test ----------------
status("Step 5/5: TRAINING FINISHED. Loading best model and testing...")
model.load_state_dict(torch.load(f"{TAG}_best.pth"))
Pv, Yv = probs(dl_va); Pt, Yt = probs(dl_te)
status("Tuning per-class thresholds on validation set...")
ths = []
for c in range(5):
    grid = np.arange(0.05, 0.96, 0.02)
    ths.append(grid[np.argmax([f1_score(Yv[:,c], Pv[:,c]>=t, zero_division=0) for t in grid])])
pred = (Pt >= np.array(ths)).astype(int)
per = [f1_score(Yt[:,c], pred[:,c], zero_division=0) for c in range(5)]
status(f"[{MODE}] TEST Macro-F1: {np.mean(per):.4f}")
print(dict(zip(LABELS, np.round(per, 4))), flush=True)
np.save(f"{TAG}_val_probs.npy", Pv); np.save(f"{TAG}_test_probs.npy", Pt)
np.save(f"{TAG}_test_labels.npy", Yt); np.save(f"{TAG}_thresholds.npy", np.array(ths))
status("ALL DONE. Files saved: " + ", ".join(f"{TAG}_{n}" for n in ["best.pth","val_probs.npy","test_probs.npy","test_labels.npy","thresholds.npy"]))

[STATUS   0.0 min] STARTING | mode=xattn_query | cnn=resnet50 | epochs=30 | gpu=Tesla T4
[STATUS   0.0 min] Step 1/5: reading CSV and checking that all image files exist
[STATUS   0.1 min] Data OK | train 5881 | val 1260 | test 1260
[STATUS   0.1 min] Step 2/5: building model (RETFound + LoRA, ResNet50, fusion layers)
trainable params: 1,572,864 || all params: 304,874,496 || trainable%: 0.5159


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

[STATUS   0.1 min] Model ready | trainable params: 33.0 M
[STATUS   0.1 min] Step 3/5: setting up loss and optimizer
[STATUS   0.1 min] Step 4/5: TRAINING STARTED (30 epochs, expect about 3-4 min per epoch)
[STATUS   0.2 min]   epoch 1/30 | batch 0/368 | loss 0.3306
[STATUS   0.5 min]   epoch 1/30 | batch 50/368 | loss 0.1100
[STATUS   0.9 min]   epoch 1/30 | batch 100/368 | loss 0.1264
[STATUS   1.2 min]   epoch 1/30 | batch 150/368 | loss 0.1270
[STATUS   1.6 min]   epoch 1/30 | batch 200/368 | loss 0.1293
[STATUS   2.0 min]   epoch 1/30 | batch 250/368 | loss 0.1137
[STATUS   2.3 min]   epoch 1/30 | batch 300/368 | loss 0.0981
[STATUS   2.7 min]   epoch 1/30 | batch 350/368 | loss 0.1314
[STATUS   2.8 min]   epoch 1/30 | training done, validating...
[STATUS   3.1 min] EPOCH 1/30 DONE | loss 0.1621 | val Macro-F1 0.3398 | best 0.3398 | 3.0 min/epoch | ~86 min left (max)
[STATUS   3.1 min]   new best model saved (val Macro-F1 0.3398)
[STATUS   3.1 min]   epoch 2/30 | batch 0/368 | los